# Notebook 03 — Mechanism ablations and protocol freeze

**Purpose.** Byte- and exposure-matched ablations A1 (foveation), A2 (learned gate), P (disagreement auxiliary) and the exploratory P+MSF; comparator selection on tune; development paired-loss SD and MDE scenarios; leakage audit rerun; protocol lock. **Partitions:** train, tune. **GPU:** ablation pool. This notebook may reject the proposed method; rejection is a valid result.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


## Ablation fits (seed 101, identical schedule and augmentation)

In [ ]:
for cfg in ['A1', 'A2', 'P', 'P_MSF']:
    run(['train.py', '--config', cfg, '--phase', 'dev', '--seed', '101', '--stage', 'ablations'])
    rd = sorted(ws.runs.glob(f'dev_{cfg}_s101_*'), key=lambda p: p.stat().st_mtime)[-1]
    run(['predict.py', '--run', rd.name, '--partitions', 'tune', '--views', 'full'])
# common shorter-schedule pilots (configs 7 and 8 of the eight allowed): complete 3-epoch cosine schedule for P and the pretrained comparator
for cfg in ['P', 'B3']:
    run(['train.py', '--config', cfg, '--phase', 'dev', '--seed', '101', '--epochs', '3', '--stage', 'ablations', '--tag', 'ep3'])
    rd = sorted(ws.runs.glob(f'dev_{cfg}_s101_*'), key=lambda p: p.stat().st_mtime)[-1]
    run(['predict.py', '--run', rd.name, '--partitions', 'tune', '--views', 'full'])

## Development table, comparator selection, feasibility scenarios
All compact variants and the pretrained comparator reach their best tune loss within the first three epochs of the 12-epoch schedule, so the allowed *common shorter epoch count* pilot (a complete 3-epoch cosine schedule) is run for P and B3. Selection is restricted to runs trained with that complete schedule, because the final refit must run a complete schedule without tune-based checkpoint selection.

In [ ]:
run(['evaluate_dev.py', '--schedule-epochs', '3'])
cand = read_json(ws.evaluation / 'development' / 'protocol_candidate.json'); print(json.dumps({k: cand.get(k) for k in ['comparator','candidate_run','final_epochs','development_paired_sd','mde_test_at_dev_sd','mde_scenarios','referral_score','referral_score_aurc_tune','n_development_configs']}, indent=1))

## Input-only feature audit and parameter counts

In [ ]:
from cape_eeg.model import build_model, count_parameters, CONFIGS
for cid in list(CONFIGS) + ['B3']:
    print(cid, count_parameters(build_model(cid)) if cid != 'B3' else 'see notebook 02')
print('gate inputs: [uL, uC, valid_L, valid_C, JS(pL,pC)] - no patient id, vote count or label enters the network at inference')

## Leakage audit rerun and protocol lock
The lock is written once; it binds the comparator, epochs, referral score, strata, corruption list and figure list before any test access.

In [ ]:
run(['make_splits.py'])  # deterministic: must reproduce the identical split hash
assert read_json(ws.manifests / 'leakage_audit.json')['status'] == 'PASS'
if not (ws.manifests / 'protocol_lock.json').exists():
    run(['evaluate_dev.py', '--schedule-epochs', '3', '--lock'])
lock = read_json(ws.manifests / 'protocol_lock.json'); print('protocol_hash', lock['protocol_hash'], '| comparator', lock['comparator'], '| final epochs', lock['final_epochs'], '| referral score', lock['referral_score'])